# AMR-RL Adversarial Co-Training — Google Colab
Run cells top-to-bottom for a full training run, or execute individual cells selectively.
Replace `REPO_URL` in the first cell before starting.

In [ ]:
# ── 1. Clone repo ────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/YOUR_USERNAME/amr_rl.git"  # <-- replace

import os
if not os.path.isdir("amr_rl"):
    !git clone {REPO_URL}
%cd amr_rl

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
# torch is pre-installed on Colab GPU runtimes; pip keeps the existing CUDA
# build when the pyproject.toml constraint is already satisfied.
!pip install -e '.[notebooks]' --quiet

In [ ]:
# ── 3. GPU check ─────────────────────────────────────────────────────────────
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device:       {device}")
if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"VRAM:         {props.total_memory / 1e9:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("No GPU — training will be slow on CPU.")

In [ ]:
# ── 4. Download / verify data ─────────────────────────────────────────────────
# Dry-run by default: reports stats on the committed data files.
# Swap to the second command to re-download fresh data from BV-BRC.
!python scripts/download_real_patric_data.py --dry_run
# !python scripts/download_real_patric_data.py   # <- uncomment for real download

In [ ]:
# ── 5. Pretrain resistance model ──────────────────────────────────────────────
# Uses synthetic mock data (no download required). Output:
#   checkpoints/resistance_pretrained.pt
#   checkpoints/ec50_predictor.pt
!python -m resistance.pretraining.pretrain_resistance \
    --mock \
    --epochs 30 \
    --device {device} \
    --output checkpoints/resistance_pretrained.pt

In [ ]:
# ── 6. Train fixed-resistance baseline ───────────────────────────────────────
# PPO against a static (non-adaptive) resistance model — key paper baseline.
# Output: checkpoints/fixed_ppo_static.zip
!python -m training.agents.fixed_resistance_agent \
    --resistance_mode static \
    --total_timesteps 2000000 \
    --seed 42 \
    --device {device}

In [ ]:
# ── 7. Adversarial co-training — seed 42 ─────────────────────────────────────
!python scripts/train.py \
    --config config.yaml \
    --total_timesteps 2000000 \
    --seed 42 \
    --device {device} \
    --ec50_predictor checkpoints/ec50_predictor.pt

In [ ]:
# ── 8. Adversarial co-training — seed 7 ──────────────────────────────────────
!python scripts/train.py \
    --config config.yaml \
    --total_timesteps 2000000 \
    --seed 7 \
    --device {device} \
    --ec50_predictor checkpoints/ec50_predictor.pt

In [ ]:
# ── 9. Adversarial co-training — seed 21 ─────────────────────────────────────
!python scripts/train.py \
    --config config.yaml \
    --total_timesteps 2000000 \
    --seed 21 \
    --device {device} \
    --ec50_predictor checkpoints/ec50_predictor.pt

In [ ]:
# ── 10. Evaluate ──────────────────────────────────────────────────────────────
# Uses seed-42 checkpoints as the primary policy.
# Adjust --policy_path / --training_history if a different seed produced the best run.
!python scripts/evaluate.py \
    --config config.yaml \
    --policy_path checkpoints/best/best_model.zip \
    --fixed_policy_path checkpoints/fixed_static_best/best_model.zip \
    --adversary_path checkpoints/adversary_final.pt \
    --training_history runs/training_history.json \
    --output_dir results/ \
    --n_episodes 200

In [ ]:
# ── 11. Download results ──────────────────────────────────────────────────────
from google.colab import files
import zipfile, pathlib

with zipfile.ZipFile("amr_rl_outputs.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["results", "checkpoints", "runs"]:
        for p in pathlib.Path(folder).rglob("*"):
            if p.is_file():
                zf.write(p)

files.download("amr_rl_outputs.zip")